### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [14]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

### Start with our Message class

In [15]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [16]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [17]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [18]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [19]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [20]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [21]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [22]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some compelling reasons to choose AutoGen for your AI Agent project:

1. **Improved Efficiency**: AutoGen employs asynchronous messaging and an event-driven architecture, which allows for faster and more efficient communication between agents. This can lead to quicker response times and overall better performance in complex tasks.

2. **Scalability**: The modularity and extensibility of AutoGen enable the creation of scalable and customizable systems. This ensures that as your project grows, you can easily adapt and expand your agent capabilities without significant overhead.

3. **Flexibility**: AutoGen supports a variety of use cases and can be tailored to meet specific project requirements, making it suitable for diverse applications in different industries.

4. **Integration Capabilities**: AutoGen can be easily integrated with existing systems and APIs, facilitating seamless interaction with third-party services and data sources.

5. **Community and Support**: Being part of a larger ecosystem usually means better community support, more available resources, and shared knowledge, which can be invaluable during development.

These advantages position AutoGen as a strong candidate for enhancing the efficiency, scalability, and flexibility of AI Agent projects.

TERMINATE

## Cons of AutoGen:
Here are some reasons against choosing AutoGen for a new AI Agent project:

1. **Difficult Documentation**: The documentation for AutoGen is often cited as hard to read and lacking sufficient examples, making it challenging for developers to effectively use the tool.

2. **Functionality Issues**: Certain features, such as structured outputs, reportedly do not function as expected, which could lead to implementation problems and hinder project progress.

3. **Lack of Differentiation**: AutoGen may not clearly differentiate itself from similar applications, like AG2, which could lead to confusion in selecting the right tool for your needs.

4. **Complexity in Agent Delegation**: While agent delegation is a touted benefit, it may come with its own complexities that could complicate project implementation and require deeper understanding and expertise.

These factors should be carefully considered when deciding whether to incorporate AutoGen into your project. 

TERMINATE



## Decision:

Based on the research provided by the team, I would recommend proceeding with AutoGen for the AI Agent project despite its drawbacks. The significant advantages, such as improved efficiency, scalability, flexibility, and integration capabilities, outweigh the concerns regarding documentation and functionality issues. The potential to adapt and grow within a supportive community also enhances its appeal, making AutoGen a compelling choice to enhance the overall performance of the project.

TERMINATE

In [23]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [24]:
await host.stop()

In [1]:
# AutoGen Practice By Lee McCormick
# Lab4 : AutoGen Core Distributed

from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown
from dotenv import load_dotenv
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

load_dotenv(override=True)

# 📝 AutoGen --> AutoGen Core Distributed
display(Markdown("📝 📝 📝 AutoGen -->  AutoGen Core Distributed 📝 📝 📝"))

ALL_IN_ONE_WORKER = True

@dataclass
class Message:
    content: str

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)
    

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:
    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")

response = await worker.send_message(Message(content="Go!"), agent_id)
display(Markdown(response.content))
display(Markdown("⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️"))


📝 📝 📝 AutoGen -->  AutoGen Core Distributed 📝 📝 📝

## Pros of AutoGen:
Here are some reasons in favor of choosing AutoGen for your AI Agent project:

1. **Scalability**: AutoGen's modular design allows for the creation of scalable and customizable systems, making it easy to adjust to growing demands or complexity in your AI applications.

2. **Ease of Use**: The framework includes integrated observability and debugging tools, simplifying the monitoring and control of agent workflows. This can significantly reduce development time and enhance productivity.

3. **Modularity**: AutoGen's architecture enables you to pick and choose components, allowing for tailored solutions that meet specific project requirements without unnecessary overhead.

4. **Extensibility**: The framework is designed to be extendable, making it easier to integrate new features or technologies as they emerge.

5. **Community and Support**: Leveraging AutoGen may provide access to a community of users and resources which can be beneficial for troubleshooting and sharing best practices.

These benefits could make AutoGen a valuable asset in your AI Agent project. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen for an AI Agent project:

1. **Steep Learning Curve**: Many users find that mastering AutoGen requires significant time and effort due to its complexity.

2. **Limited Creativity**: While effective at generating content based on templates, AutoGen may lack the innovative creative capabilities needed for more dynamic applications.

3. **Suitability for Applications**: Although AutoGen is beneficial for research and prototypes, it may not be ideal for customer-facing applications where human-like interaction is crucial.

4. **High Development Costs**: For small businesses or projects with tight budgets, the cost of developing and maintaining AutoGen agents may be prohibitive compared to the potential benefits.

5. **Complexity in Integration**: Integrating AutoGen into existing systems can be challenging due to its versatile but intricate configurations, potentially discouraging its use in simpler projects.

These factors could be important to consider when deciding whether to implement AutoGen in your project. 

TERMINATE



## Decision:

Based on the research provided, I recommend using AutoGen for your AI Agent project. The key advantages, such as scalability, ease of use, modularity, and extensibility, present a strong case for its implementation. These benefits can significantly enhance productivity and allow for tailored solutions that adapt to changing project requirements. 

While the cons, including the steep learning curve and complexity in integration, are notable, they are manageable challenges that can be addressed with proper training and planning. The access to a supportive community further mitigates potential obstacles. Therefore, the overall potential for innovation and efficiency that AutoGen offers outweighs its drawbacks. 

TERMINATE

⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️⏭️